# Séquence des TP TALL avec bilbiothèque progressivement constituée

# <span style="color:red">Push avant de quitter !!<span>

## TP 1: Constitution de corpus (OLGA) - Semaine 21 Janvier ?

### Descriptif
Problème à résoudre: on veut constituer un corpus pour répondre à une question de recherche (exemple emergence du genre scifi). Première série de manips: statistiques descriptives sur les méta-données d'un grand corpus (chapitre/gutenberg). Pas mal de nettoyage à envisager. 

Deuxième manip du même TP: on part d'un corpus propre comme cidre ou girls et on formule une question de recherche : par exemple bow + spécificité/tf-idf et évolution du vocabulaire spécifique par auteur ou dans le temps. 

Bibliothèque: montée en mémoire à partir d'un fichier texte, tokenisation (brutale), calcul bow, calcul tf-idf

### Enoncé

### Corrigé complet

### Code à trous

### Fonctions pour bibliothèque

In [1]:
# Fonction de chargement du corpus
import nltk
# Si nécessaire, télécharger punkt_tab
nltk.download('punkt_tab')

def charge_corpus(filename):
    ''' retourne une liste de phrases
    qui sont elles-même des listes de tokens'''
    corpus = []
    with open(filename, "r") as f:
        for p in nltk.tokenize.sent_tokenize(f.read()):
            corpus.append(nltk.word_tokenize(p))
    return corpus

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/amsili/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
# Test de la fonction de chargement de corpus
path = "Corpora/Candide.txt"
corpus = charge_corpus(path)
print(len(corpus))
print(" ".join(corpus[5]))

1522
Sa grande salle même était ornée d'une tapisserie .


## TP 2 Classification supervisée sur les documents du corpus (ANTOINE)

features: bow; tâche: attribution d'auteur ; 

Possibilité de faire l'implémentation de naïve bayes; utilisation de scikit learn; envisager peut-être aussi d'analyse d'erreur

### Énoncé

Sur quel corpus on travaille ?

Quelles features ? 

Quelle classification ? pourquoi  tant de haine ?

## TP 3 Evaluation de la classif (rappel/précision) (PASCAL)

analyse d'erreur, comparaison de quelques algos de classif; différentes jeu de traits (td-idf...)

Evaluation k-fold pour évaluer la variance 

## TP 4 Programmation d'un n-gramme de mots (OLGA)

Tâche possible: comparaison de différentes tailles de fenêtre; application à la detection d'auteurs par perplexité (un modèle par auteur)

(PASCAL : je place ici un TP que j'ai déjà utilisé)

### Modèle de base

#### Construction de la structure de données

Avec le corpus français de votre choix, pris par exemple dans lacollection gutemberg, segmenté en phrases (délimitées par les pseudo tokens `<s>` et `</s>` et tokenisé (avec spacy ou avec nltk), pour une valeur de $n$ fixée, remplir un dictionnaire python (faisant le décompte pour chaque $(n-1)$-gramme de tous les mots qui apparaissent après ce $(n-1)$-gramme (et son nombre d'occurrences).

In [3]:
def construit_modele(corpus, n=2):
    modele = {}
    for phrase in corpus:
        phrase = ['<s>'] + phrase + ['</s>']
        for i in range(len(phrase) - n + 1):
            contexte, mot = tuple(phrase[i:i+n-1]), phrase[i+n-1]
            if contexte in modele.keys():
                if mot in modele[contexte].keys():
                    modele[contexte][mot] += 1
                else:
                    modele[contexte][mot] = 1
            else:
                modele[contexte] = { mot : 1 }
    return modele

In [4]:
# Test avec Candide
modele = construit_modele(charge_corpus("Corpora/Candide.txt"))
# Les clés du dictionnaire sont des tuples de mots, même pour n=2
print(len(modele))
clefs = [('<s>',), ('la',), ('petite',)]
for cle in clefs:
    print("contexte: %12s : %s" %(cle,list(modele[cle])[:6]))
print(modele[('est',)])

5950
contexte:     ('<s>',) : ['CANDIDE', 'Sa', 'Il', 'Les', 'Monsieur', 'Tous']
contexte:      ('la',) : ['nature', 'maison', 'soeur', 'Westphalie', 'baronne', 'rendait']
contexte:  ('petite',) : ['brune', 'porte', 'ville', 'armée', 'rivière', 'barque']
{'démontré': 1, 'nécessairement': 1, 'bien': 15, 'au': 5, 'assez': 2, 'faite': 3, 'assurée': 1, 'regardé': 1, 'enchaîné': 1, 'morte': 3, 'peut-être': 2, 'évidemment': 1, 'particulière': 1, 'admirable': 1, 'précipité': 1, 'englouti': 1, 'démontrée': 1, 'vrai': 7, 'ce': 1, 'impossible': 3, 'un': 1, 'arrivé': 1, 'le': 6, 'encore': 2, 'mon': 2, 'venue': 1, 'amoureux': 1, 'plus': 2, 'presque': 1, 'ma': 1, 'dur': 1, 'commune': 1, 'à': 6, 'fort': 2, 'divisé': 1, 'allemand': 2, 'il': 1, 'une': 2, 'pleine': 1, "l'inhumanité": 1, 'peu': 2, 'bonne': 1, 'donc': 2, "d'une": 1, "l'ancienne": 1, 'certain': 1, "d'embrasser": 1, 'pas': 1, 'passablement': 1, 'fait': 1, 'mal': 1, 'la': 1, 'votre': 1, 'que': 1, 'puni': 1, 'folle': 1, 'trop': 1, 'communéme

#### Fonction de prédiction

À partir de ces décomptes (comptages bruts d'occurrences), on peut écrire une fonction de prédiction, qui, étant donné un contexte (on acceptera un contexte de longueur quelconque mais la fonction ne
prendra en compte que les $n-1$ derniers mots), renvoie le mot qui a le score le plus élevé (sans passer par les probabilités).  Cette fonction ne peut pas toujours faire de prédiction, puisqu'il peut arriver que $n-1$-gramme pertinent dans le contexte n'ai jamais été rencontré. Dans ce cas, on prédira conventionnellement le token `XXX`.

In [5]:
def prediction_greedy(modele, n, contexte):
    '''prédit le mot qui doit suivre en prenant
    les derniers n-1 mots du contexte (si possible)'''
    if len(contexte) >= n-1:
        cle = tuple(contexte[-(n-1):])
        if cle in modele.keys():
            choix = modele[cle]
            return max(choix, key=choix.get)
    return 'XXXX'

In [7]:
def verification_prediction(m,N):
    for w in ['<s>', 'Le', 'Candide', 'petite']:
        contexte = (w,)
        print("Après le mot %s le modèle prédit %s." % (w,prediction_greedy(modele,N,contexte)))
verification_prediction(modele,2)

Après le mot <s> le modèle prédit --.
Après le mot Le le modèle prédit lendemain.
Après le mot Candide le modèle prédit ,.
Après le mot petite le modèle prédit métairie.


#### Evaluation intrinsèque, exactitude
En prenant des corpus de test de 200 à 500 occurrences, calculer le score d'exactitude du modèle avec prédiction greedy sur:
1. un texte tiré du corpus d'apprentissade
2. un texte du même auteur mais non vu à l'apprentissage
3. un texte complètement différent (e.g., wikipedia)

On va supposer que le format du corpus de test est le même que celui du corpus initial (liste de listes de mots). 

In [27]:
def mesure_exactitude(test,modele,n=2):
    hits = 0
    miss = 0
    for phrase in test:
        phrase = ['<s>'] + phrase + ['</s>']
        for i in range(len(phrase) - n + 1):
            contexte, mot = tuple(phrase[i:i+n-1]), phrase[i+n-1]
            if prediction_greedy(modele, n, contexte) == mot:
                hits += 1
            else:
                miss +=1
    return hits / (hits + miss)
            

In [28]:
print(mesure_exactitude(corpus, modele))

0.320610298792117


In [29]:
# Création d'autres corpus de test
voltaire = charge_corpus("Corpora/extrait_Voltaire-oeuvre_500w.txt")

In [30]:
print(mesure_exactitude(voltaire,modele))

0.0867579908675799


In [31]:
# Troisième corpus de test
wikipedia = charge_corpus("Corpora/extrait_Wikipedia-menthe_500w.txt")

In [33]:
print(mesure_exactitude(wikipedia,modele))

0.07648183556405354


### Repli (backoff), lissage (smoothing), perplexité
Pour cette deuxième partie on prend comme valeur de $n$ la valeur 3.
#### Repli
Au lieu de répondre `XXX` quand on n'a jamais rencontré le (n-1)-gramme, on peut implémenter une stratégie de repli, qui consiste à considérer comme contexte le (n-2)-gramme, s'il a déjà été rencontré, et ainsi de suite jusqu'à l'unigramme. Définir et évaluer le nouveau modèle ainsi obtenu. 

In [55]:
# Il faut modifier la construction du modèle qui doit faire des comptages pour toutes les valeurs 
# de n (y compris unigrammes), et bien sûr proposer une nouvelle fonction de prédiction.
# Les unigrammes sont les mots qui apparaissent dans un contexte vide. La sdd est compatible. 
def construit_modele_repli(corpus, n=3):
    modele = {}
    for phrase in corpus:
        phrase = ['<s>'] + phrase + ['</s>']
        for i in range(len(phrase) - n + 1):
            k = n
            while k >= 1:
                contexte, mot = tuple(phrase[i:i+k-1]), phrase[i+k-1]
                if contexte in modele.keys():
                    if mot in modele[contexte].keys():
                        modele[contexte][mot] += 1
                    else:
                        modele[contexte][mot] = 1
                else:
                    modele[contexte] = { mot : 1 }
                k -= 1
    return modele

In [95]:
modele_complet = construit_modele_repli(charge_corpus("Corpora/Candide.txt"),4)

In [96]:
print(modele_complet[()]['le'])
print(modele_complet[('la','petite')])

671
{'ville': 1, 'armée': 1, 'métairie': 1, 'société': 1}


In [97]:
def prediction_greedy_repli(modele, n, contexte):
    '''prédit le mot qui doit suivre en prenant
    les derniers n-1 mots du contexte (si possible)
    et en appliquant du backoff si nécessaire'''
    l = len(contexte)
    k = min(l, n-1)
    while k >= 1:
        cle = tuple(contexte[-(k-1):])
        print(cle, "k = %d" % k)
        if cle in modele.keys():
            choix = modele[cle]
            return max(choix, key=choix.get)
        k -= 1
    return 'XXXX'

In [98]:
prediction_greedy_repli(modele_complet, 3, ['la', 'petite'])

('petite',) k = 2


'métairie'

In [99]:
phrase = ['<s>', 'Monsieur', 'le', 'chignon', 'était', 'un', 'des', 'plus',
    'puissants', 'seigneurs', 'de', 'la', 'Westphalie', ',', 
    'car', 'son', 'château', 'avait', 'une', 'porte', 
    'et', 'des', 'fenêtres']
for i in range(len(phrase)-5):
    contexte = tuple(phrase[i:i+4])
    prediction_greedy_repli(modele_complet, 4, contexte)
    


('le', 'chignon') k = 3
('chignon',) k = 2
('<s>', 'Monsieur', 'le', 'chignon') k = 1
('chignon', 'était') k = 3
('était',) k = 2
('était', 'un') k = 3
('un', 'des') k = 3
('des', 'plus') k = 3
('plus', 'puissants') k = 3
('puissants', 'seigneurs') k = 3
('seigneurs', 'de') k = 3
('de', 'la') k = 3
('la', 'Westphalie') k = 3
('Westphalie', ',') k = 3
(',', 'car') k = 3
('car', 'son') k = 3
('son', 'château') k = 3
('château', 'avait') k = 3
('avait', 'une') k = 3
('une', 'porte') k = 3
('porte', 'et') k = 3


In [89]:
corpus = charge_corpus("Corpora/Candide.txt")
for phrase in corpus[4:5]:
    phrase = ['<s>'] + phrase + ['</s>']
    for i in range(len(phrase) - 5 + 1):
        for j in range(5):
            contexte = tuple(phrase[i:i+j-1])
            prediction_greedy_repli(modele, 3, contexte)

('.',) k = 2
('<s>', 'Monsieur', 'le', 'baron', 'était', 'un', 'des', 'plus', 'puissants', 'seigneurs', 'de', 'la', 'Westphalie', ',', 'car', 'son', 'château', 'avait', 'une', 'porte', 'et', 'des', 'fenêtres', '.') k = 1
('<s>',) k = 1
('Monsieur',) k = 2
('<s>', 'Monsieur') k = 1
('le',) k = 2
('<s>', 'Monsieur', 'le') k = 1
('Monsieur',) k = 1
('le',) k = 2
('Monsieur', 'le') k = 1
('baron',) k = 2
('Monsieur', 'le', 'baron') k = 1
('le',) k = 1
('baron',) k = 2
('le', 'baron') k = 1
('était',) k = 2
('le', 'baron', 'était') k = 1
('baron',) k = 1
('était',) k = 2
('baron', 'était') k = 1
('un',) k = 2
('baron', 'était', 'un') k = 1
('était',) k = 1
('un',) k = 2
('était', 'un') k = 1
('des',) k = 2
('était', 'un', 'des') k = 1
('un',) k = 1
('des',) k = 2
('un', 'des') k = 1
('plus',) k = 2
('un', 'des', 'plus') k = 1
('des',) k = 1
('plus',) k = 2
('des', 'plus') k = 1
('puissants',) k = 2
('des', 'plus', 'puissants') k = 1
('plus',) k = 1
('puissants',) k = 2
('plus', 'puissants')

## TP 5 POS-tagging (MATHIEU + ANTOINE)
Manip: règles à la main (voir TP Mathieu) pour faire émerger des features; 

(puis manips avec scikit learn)

## TP 6 modèles n-gramme de POS-tag (OLGA)
mesure du gain par rapport au TP4, et tâche d'identification de la date d'écriture avec régression linéaire

## TP 7 TP de Mathieu sur la réduction de dim. pays et capitales (MATHIEU + OLGA)
à adapter (ou pas) aux romans/personnages/auteurs 

## TP 8 plongements lexicaux: matrice terme-terme + utilisation de distances cosinus (PASCAL)
Tâche : recherche de biais par analogie, par exemple "roi" - "homme" + "femme" = "reine".

## TP 9 Perceptron (PASCAL)

### Descriptif
tâche = reconnaître la fonction logique "ET". Le TP est déjà prêt. 


### Enoncé
On travaille dans ce TP avec un perceptron simple à deux entrées, avec la fonction de Heaviside comme fonction de transfert, qui réalise une classification binaire ($\hat{y} \in \{0, 1\}$), et on va étudier les capacités de ce perceptron d'apprendre les fonctions booléennes **and**, **or**, **nor**, ou **xor**.

### and

| $x_1$ | $x_2$ | $y$ |
|-------|-------|-----|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

### or

| $x_1$ | $x_2$ | $y$ |
|-------|-------|-----|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 1 |

### nor

| $x_1$ | $x_2$ | $y$ |
|-------|-------|-----|
| 0 | 0 | 1 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 0 |

### xor

| $x_1$ | $x_2$ | $y$ |
|-------|-------|-----|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

1. En supposant un neurone simple à 2 entrées et un biais, et des poids déjà initialisés (on prendra les valeurs $(-1, 1.4, 1)$ pour commencer), écrire une fonction `prediction()` qui retourne la classe prédite par le perceptron.

2. Écrire une fonction `epoque()` qui, pour chacun des triplets d'un ensemble d'apprentissage, calcule la prédiction faite par le neurone, calcule l'erreur et modifie les poids en appliquant la règle de mise à jour vue en classe :

   $$
   \Delta a_j = \mu \, (y - \hat{y}) \, x_j
   $$

   On va supposer que $\mu = 0.1$.

3. Écrire une fonction d'apprentissage qui initialise les poids, fait un apprentissage en $n$ époques (on suppose $n = 5$) et affiche la valeur de l'erreur globale.

4. Faire en sorte que la fonction `epoque()` calcule une erreur globale, et définir une nouvelle fonction d'apprentissage qui arrête le calcul une fois que l'erreur globale est nulle.

5. Ajouter une initialisation aléatoire des poids (par exemple avec des valeurs comprises entre $-1$ et $1$, puis avec des valeurs comprises entre $-100$ et $100$), et étudier la convergence de l'apprentissage.

6. Étudier la façon dont le taux d'apprentissage peut influer sur la vitesse de convergence.

Une combinaison donnée de 3 poids correspond à une droite séparatrice. Par exemple, les poids $(-1, 1.4, 1)$ correspondent à l'équation

$$
-1 + 1.4x + y = 0
$$

c'est-à-dire

$$
y = -1.4x + 1.
$$

On peut donc assez facilement afficher la succession des droites produites à chaque époque pour visualiser la façon dont le calcul aboutit à une droite séparatrice.

1. En supposant un neurone simple à 2 entrées et un biais, et des poids déjà initialisés (on prendra les valeurs $(-1, 1.4, 1)$
 pour commencer), écrire une fonction prediction() qui retourne la classe prédite par le perceptron.

In [4]:
# ----------------------------------------------------------------------
# Fonction : prediction()
# Prototype: entrée: x_1 et x_2
#            poids: triplets de poids
#            sortie: valeur prédite par le perceptron défini par ces poids
# Contient : applique le calcul de prédiction
#            (combinaison linéaire poids x entrée + fonction de transfert)
# ----------------------------------------------------------------------
def prediction(x1, x2, poids):
    somme = poids[0] + x1 * poids[1] + x2 * poids[2]
    activ = 0 if somme <= 0 else 1
    return activ

### Corrigé complet

In [2]:
# ---------------------------------------- -*- coding: utf-8 -*---------
# Fichier  : perceptron_ng0.py
# Contient : TP sur le perceptron, version non graphique
#            
# Dépend   : random pour l'initialisation des poids
# --
# ----------------------------------------------------------------------
import random

# ----------------------------------------------------------------------
# Fonction : prediction()
# Prototype: entrée: x_1 et x_2
#            poids: triplets de poids
#            sortie: valeur prédite par le perceptron défini par ces poids
# Contient : applique le calcul de prédiction
#            (combinaison linéaire poids x entrée + fonction de transfert)
# ----------------------------------------------------------------------
def prediction(x1, x2, poids):
    somme = poids[0] + x1 * poids[1] + x2 * poids[2]
    activ = 0 if somme <= 0 else 1
    return activ

# ----------------------------------------------------------------------
# Fonction : epoque()
# Prototype: entrée: ref (liste de triplets d'apprentissage)
#            poids: triplets de poids modifiés par effet de bord
#            sortie: /
# Contient : boucle sur toutes les données de la référence,
#            calcule l'erreur et modifie les poids passés en paramètre
# --
# La version _g calcule en plus le score global d'erreur (somme des
# valeurs absolues des erreurs), et renvoie cette valeur
# ----------------------------------------------------------------------
def epoque(ref, poids, nu):
    for x1, x2, y in ref:
        erreur = y - prediction(x1, x2, poids)
        if erreur:
            poids[0] += nu * erreur
            poids[1] += nu * erreur * x1
            poids[2] += nu * erreur * x2

def epoque_g(ref, poids, nu):
    erreur_globale = 0
    for x1, x2, y in ref:
        erreur = y - prediction(x1, x2, poids)
        erreur_globale += abs(erreur)
        if erreur:
            poids[0] += nu * erreur
            poids[1] += nu * erreur * x1
            poids[2] += nu * erreur * x2
    return erreur_globale

# ----------------------------------------------------------------------
# Fonction : erreur_globale()
# Prototype: entrée: ref (liste de triplets d'apprentissage)
#            poids: triplets de poids modifiés par effet de bord
#            sortie: erreur globale pour un ensemble de prédictions
# Contient : boucle sur toutes les données de la référence,
#            cumule les valeurs absolues d'erreur pour chaque prédiction
# ----------------------------------------------------------------------
def erreur_globale(ref, poids):
    erreur_globale = 0
    for x1, x2, y in ref:
        erreur = y - prediction(x1, x2, poids)
        erreur_globale += abs(erreur)
    return erreur_globale


# ----------------------------------------------------------------------
# Fonction : apprentissage_XXX()
# Prototype: entrée: ref (liste de triplets d'apprentissage)
#            sortie: /
#            effet de bord: affichage résultats apprentissage
# Contient : plusieurs variantes de fonction qui lancent l'apprentissage
#            après avoir initialisé les poids
#      simple: nombre d'époques prédéfinie, affichage erreur globale
#      nmax: nombre d'époques maximales prédéfinie, stop si convergence
#      conv: arrêt de l'apprentissage dès qu'il y a convergence
#            (avec une borne (1000) sur le nombre d'époques)
# ----------------------------------------------------------------------
def apprentissage_simple(ref,n=5):
    poids = init_poids()
    print(poids)
    for i in range(n):
        epoque(ref,poids,nu)
    print("Après %d époques, erreur globale: %d" % (n, erreur_globale(ref,poids)))

def apprentissage_nmax(ref,n = 5):
    poids = init_poids()
    print(poids)
    for i in range(n):
        e = epoque_g(ref,poids,nu)
        if not e:
            print("Epoque n°%d: convergence !" % (i+1), poids)
    print("Après %d époques, erreur globale = %d" % (n,e))

def apprentissage_conv(ref):
    poids = init_poids()
    print(poids)
    c = 0
    while (1):
        c += 1
        e = epoque_g(ref,poids,nu)
        if e == 0:
            print("Epoque n°%d: convergence !" % (c+1), poids)
            break
        if c > 1000:
            print("Après %d époques, erreur globale = %d" % (c,e))
            break



# **********************************************************************
# VARIABLES GLOBALES & PARAMÈTRES
# Données de référence: fonction qu'on cherche à approximer
# (x1, x2, y)
#      x1, x2 \in {0, 1}
#      y (classe) \in {0, 1}
AND = [[0,0,0],[0,1,0],[1,0,0],[1,1,1]]
OR = [[0,0,0],[0,1,1],[1,0,1],[1,1,1]]
NOR = [[0,0,1],[0,1,0],[1,0,0],[1,1,0]]
XOR = [[0,0,0],[0,1,1],[1,0,1],[1,1,0]] # Non linéairement séparable
data_ref = AND

# Taux d'apprentissage
nu = .1

# Les poids qui représentent les propriétés du neurone
# def init_poids():
#     return [-1, 1.4, -1] 
def init_poids():
    poids = []
    for _ in range(3):
        poids.append(2 * random.random() - 1)
    return poids
    

# **********************************************************************
# PROGRAMME PRINCIPAL
    
# apprentissage_simple(data_ref, 10)
# apprentissage_nmax(data_ref, 10)
apprentissage_conv(data_ref)


[-0.6068949547601763, 0.010546310308211071, 0.11975834216879688]
Epoque n°4: convergence ! [-0.40689495476017634, 0.21054631030821108, 0.3197583421687969]


### Code à trous

### Fonctions pour bibliothèque

## TP 10 Détection des mentions de personnages avec BERT (ANTOINE)
Usage de perceptron mono-couche pour prédire si l'embeddings de BERT correspond à une mention de personnage. S'il y a le temps : faire un perceptron multicouche pour faire la même tâche.  

## TP 11 Finetuning de BERT pour faire la détection de mentions des personnages (ANTOINE)

## TP 12 Interaction API LLMs modernes (PASCAL)
Tâche : correction de OCR. On prend des fichiers OCR-isés depuis Gallica. On utilise les APIs des LLMs pour faire une correction automatique (boucler sur les données en zéro shot d'abord). 


## Idées supplémentaires